In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os, json

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_auc_score,
                             roc_curve, classification_report)
from sklearn.preprocessing import label_binarize
import seaborn as sns

PLOT_DIR = r'c:\ML\EXP6\plots'
os.makedirs(PLOT_DIR, exist_ok=True)

In [ ]:
# ── 1. Load Data ──────────────────────────────────────────────────────────────
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)  # 0=malignant, 1=benign
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Class distribution:\n{y.value_counts()}")

In [ ]:
# ── 2. EDA Plots ──────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(['Malignant (0)', 'Benign (1)'], y.value_counts().sort_index(), color=['#e74c3c','#2ecc71'])
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'class_distribution.png'))
plt.savefig(os.path.join(PLOT_DIR, 'class_distribution.eps'))
plt.close()

corr = X.corr()
fig, ax = plt.subplots(figsize=(14,12))
sns.heatmap(corr, ax=ax, cmap='coolwarm', center=0, linewidths=0.3)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'correlation_heatmap.png'))
plt.savefig(os.path.join(PLOT_DIR, 'correlation_heatmap.eps'))
plt.close()
print("Saved EDA plots.")

In [ ]:
# ── 3. Train/Test Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# ── 4. Decision Tree CV Tuning ────────────────────────────────────────────────
dt_grid = [
    {'criterion': 'gini',    'max_depth': 3,    'min_samples_split': 2},
    {'criterion': 'gini',    'max_depth': 5,    'min_samples_split': 5},
    {'criterion': 'entropy', 'max_depth': 5,    'min_samples_split': 2},
    {'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2},
    {'criterion': 'gini',    'max_depth': 7,    'min_samples_split': 10},
]
dt_results = []
for p in dt_grid:
    clf = DecisionTreeClassifier(random_state=42, **p)
    acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    dt_results.append({**p, 'avg_acc': acc.mean(), 'avg_f1': f1.mean()})
    print(f"DT {p} -> acc={acc.mean():.4f}, f1={f1.mean():.4f}")

best_dt_params = max(dt_results, key=lambda r: r['avg_acc'])
print(f"\nBest DT params: {best_dt_params}")

dt_best = DecisionTreeClassifier(
    criterion=best_dt_params['criterion'],
    max_depth=best_dt_params['max_depth'],
    min_samples_split=best_dt_params['min_samples_split'],
    random_state=42)
dt_best.fit(X_train, y_train)
y_pred_dt = dt_best.predict(X_test)

# 5-fold accuracy per fold for final best model
dt_fold_scores = cross_val_score(dt_best, X, y, cv=cv, scoring='accuracy')
print(f"DT fold scores: {dt_fold_scores}")

In [ ]:
# ── 5. Random Forest CV Tuning ────────────────────────────────────────────────
rf_grid = [
    {'n_estimators': 50,  'max_depth': 5,    'max_features': 'sqrt'},
    {'n_estimators': 100, 'max_depth': 10,   'max_features': 'sqrt'},
    {'n_estimators': 100, 'max_depth': None, 'max_features': 'sqrt'},
    {'n_estimators': 200, 'max_depth': None, 'max_features': 'sqrt'},
    {'n_estimators': 100, 'max_depth': None, 'max_features': 'log2'},
]
rf_results = []
for p in rf_grid:
    clf = RandomForestClassifier(random_state=42, n_jobs=1, **p)
    acc = cross_val_score(clf, X_train, y_train, cv=cv, scoring='accuracy')
    f1  = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1')
    rf_results.append({**p, 'avg_acc': acc.mean(), 'avg_f1': f1.mean()})
    print(f"RF {p} -> acc={acc.mean():.4f}, f1={f1.mean():.4f}")

best_rf_params = max(rf_results, key=lambda r: r['avg_acc'])
print(f"\nBest RF params: {best_rf_params}")

rf_best = RandomForestClassifier(
    n_estimators=best_rf_params['n_estimators'],
    max_depth=best_rf_params['max_depth'],
    max_features=best_rf_params['max_features'],
    random_state=42, n_jobs=1)
rf_best.fit(X_train, y_train)
y_pred_rf = rf_best.predict(X_test)

rf_fold_scores = cross_val_score(rf_best, X, y, cv=cv, scoring='accuracy')
print(f"RF fold scores: {rf_fold_scores}")

In [ ]:
# ── 6. Evaluation ─────────────────────────────────────────────────────────────
def metrics(y_true, y_pred, model_proba=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, model_proba) if model_proba is not None else None
    return acc, prec, rec, f1, auc

dt_metrics = metrics(y_test, y_pred_dt, dt_best.predict_proba(X_test)[:,1])
rf_metrics = metrics(y_test, y_pred_rf, rf_best.predict_proba(X_test)[:,1])

print("\n=== FINAL RESULTS ===")
print(f"Decision Tree -> Acc:{dt_metrics[0]:.4f} Prec:{dt_metrics[1]:.4f} Rec:{dt_metrics[2]:.4f} F1:{dt_metrics[3]:.4f} AUC:{dt_metrics[4]:.4f}")
print(f"Random Forest -> Acc:{rf_metrics[0]:.4f} Prec:{rf_metrics[1]:.4f} Rec:{rf_metrics[2]:.4f} F1:{rf_metrics[3]:.4f} AUC:{rf_metrics[4]:.4f}")

In [ ]:
# ── 7. Plots ──────────────────────────────────────────────────────────────────
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm_data, title in zip(axes,
        [confusion_matrix(y_test, y_pred_dt), confusion_matrix(y_test, y_pred_rf)],
        ['Decision Tree', 'Random Forest']):
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Malignant','Benign'], yticklabels=['Malignant','Benign'])
    ax.set_title(f'{title} — Confusion Matrix')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrices.png'))
plt.savefig(os.path.join(PLOT_DIR, 'confusion_matrices.eps'))
plt.close()

# ROC Curves
fig, ax = plt.subplots(figsize=(7,6))
for model, label, color in [
        (dt_best, 'Decision Tree', '#e67e22'),
        (rf_best, 'Random Forest', '#2980b9')]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_test)[:,1])
    ax.plot(fpr, tpr, label=f'{label} (AUC={auc:.3f})', color=color)
ax.plot([0,1],[0,1],'k--')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'roc_curves.png'))
plt.savefig(os.path.join(PLOT_DIR, 'roc_curves.eps'))
plt.close()

# Feature Importance (RF)
importances = pd.Series(rf_best.feature_importances_, index=data.feature_names).sort_values(ascending=False)[:15]
fig, ax = plt.subplots(figsize=(10,6))
importances.plot.barh(ax=ax, color='#2980b9')
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'feature_importance.png'))
plt.savefig(os.path.join(PLOT_DIR, 'feature_importance.eps'))
plt.close()

print("All plots saved.")

In [ ]:
# ── 8. Save results JSON ──────────────────────────────────────────────────────
results = {
    'dt_metrics': list(dt_metrics),
    'rf_metrics': list(rf_metrics),
    'dt_fold_scores': dt_fold_scores.tolist(),
    'rf_fold_scores': rf_fold_scores.tolist(),
    'dt_results': [{k: (v if not isinstance(v, float) else round(v,4)) for k,v in r.items()} for r in dt_results],
    'rf_results': [{k: (v if not isinstance(v, float) else round(v,4)) for k,v in r.items()} for r in rf_results],
    'best_dt': {k: (v if isinstance(v, (int,str)) or v is None else v) for k,v in best_dt_params.items()},
    'best_rf': {k: (v if isinstance(v, (int,str)) or v is None else v) for k,v in best_rf_params.items()},
}
with open(r'c:\ML\EXP6\results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Results saved to results.json")